<a href="https://colab.research.google.com/github/Rikiimam16/myprojecttif/blob/main/rikz.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [24]:
print("hello mas rusdi")

print(5+5)
print(5-5)
print(5*5)
print(5/5)


hello mas rusdi
10
0
25
1.0


In [18]:
# @title ♟️ Grandmaster Chess Dashboard {display-mode: "form"}

import json
from IPython.display import HTML
from google.colab import output

def _report_js_error(message):
    print(f"JavaScript Error: {message}")

output.register_callback('report_js_error', _report_js_error)

html_content = """
<!DOCTYPE html>
<html lang='en'>
<head>
    <meta charset='UTF-8'>
    <link rel="stylesheet" href="https://unpkg.com/@chrisoakman/chessboardjs@1.0.0/dist/chessboard-1.0.0.min.css">
    <link href="https://fonts.googleapis.com/css2?family=Inter:wght@400;600;700&display=swap" rel="stylesheet">
    <style>
        body {
            font-family: 'Inter', sans-serif;
            background-color: #f0f2f5;
            margin: 0;
            padding: 20px;
            display: flex;
            justify-content: center;
        }
        .dashboard-container {
            display: grid;
            grid-template-columns: 1fr 350px;
            grid-template-rows: auto 1fr;
            gap: 20px;
            max-width: 1100px;
            width: 100%;
        }
        .kpi-row {
            grid-column: span 2;
            display: grid;
            grid-template-columns: repeat(4, 1fr);
            gap: 15px;
        }
        .card {
            background: white;
            border-radius: 12px;
            padding: 20px;
            box-shadow: 0 4px 6px rgba(0,0,0,0.05);
            border: 1px solid #e5e7eb;
        }
        .kpi-card {
            text-align: center;
        }
        .kpi-value {
            font-size: 1.5rem;
            font-weight: 700;
            color: #111827;
            margin-top: 5px;
        }
        .kpi-label {
            font-size: 0.75rem;
            color: #6b7280;
            text-transform: uppercase;
            letter-spacing: 0.05em;
        }
        .board-container {
            display: flex;
            flex-direction: column;
            align-items: center;
        }
        #myBoard {
            width: 100%;
            max-width: 600px;
            border-radius: 8px;
            overflow: hidden;
            box-shadow: 0 10px 15px -3px rgba(0,0,0,0.1);
        }
        .sidebar {
            display: flex;
            flex-direction: column;
            gap: 20px;
        }
        .history-wrapper {
            flex-grow: 1;
            display: flex;
            flex-direction: column;
            min-height: 400px;
        }
        .history-log {
            flex-grow: 1;
            background: #f9fafb;
            border: 1px solid #f3f4f6;
            border-radius: 8px;
            padding: 10px;
            font-family: 'Courier New', monospace;
            font-size: 0.95rem;
            overflow-y: auto;
            line-height: 1.6;
        }
        .controls {
            display: flex;
            gap: 10px;
            margin-top: 15px;
            width: 100%;
        }
        button {
            flex: 1;
            padding: 12px;
            border: none;
            border-radius: 8px;
            font-weight: 600;
            cursor: pointer;
            transition: all 0.2s;
        }
        .primary-btn {
            background: #22c55e;
            color: white;
        }
        .primary-btn:hover { background: #16a34a; }
        .secondary-btn {
            background: #64748b;
            color: white;
        }
        .secondary-btn:hover { background: #475569; }
        h2 { margin: 0 0 10px 0; font-size: 1rem; color: #374151; }
        .highlight-white {
            box-shadow: inset 0 0 3px 3px yellow;
        }
        .highlight-black {
            box-shadow: inset 0 0 3px 3px blue;
        }
    </style>
</head>
<body>
    <div class="dashboard-container">
        <div class="kpi-row">
            <div class="card kpi-card">
                <div class="kpi-label">Active Player</div>
                <div id="turn-kpi" class="kpi-value">White</div>
            </div>
            <div class="card kpi-card">
                <div class="kpi-label">Game Status</div>
                <div id="status-kpi" class="kpi-value">In Progress</div>
            </div>
            <div class="card kpi-card">
                <div class="kpi-label">Total Moves</div>
                <div id="moves-kpi" class="kpi-value">0</div>
            </div>
            <div class="card kpi-card">
                <div class="kpi-label">Check Status</div>
                <div id="check-kpi" class="kpi-value">None</div>
            </div>
        </div>

        <div class="card board-container">
            <div id="myBoard"></div>
            <div class="controls">
                <button class="secondary-btn" onclick="undoMove()">Undo</button>
                <button class="primary-btn" onclick="resetGame()">New Game</button>
            </div>
        </div>

        <div class="sidebar">
            <div class="card history-wrapper">
                <h2>Move History (PGN)</h2>
                <div id="pgn-history" class="history-log"></div>
            </div>
        </div>
    </div>

    <script src="https://code.jquery.com/jquery-3.5.1.min.js"></script>
    <script src="https://unpkg.com/@chrisoakman/chessboardjs@1.0.0/dist/chessboard-1.0.0.min.js"></script>
    <script src="https://cdnjs.cloudflare.com/ajax/libs/chess.js/0.10.3/chess.min.js"></script>

    <script>
        window.onerror = function(message) {
            google.colab.kernel.invokeFunction('report_js_error', [message], {});
        };

        var board = null;
        var game = new Chess();

        function onDragStart (source, piece, position, orientation) {
            if (game.game_over()) return false;
            if ((game.turn() === 'w' && piece.search(/^b/) !== -1) ||
                (game.turn() === 'b' && piece.search(/^w/) !== -1)) {
                return false;
            }
        }

        function onDrop (source, target) {
            var move = game.move({
                from: source,
                to: target,
                promotion: 'q'
            });

            if (move === null) return 'snapback';
            updateStatus();
        }

        function onSnapEnd () {
            board.position(game.fen());
        }

        function updateStatus () {
            let turn = game.turn() === 'w' ? 'White' : 'Black';
            let status = 'In Progress';
            let checkStatus = 'None';

            if (game.in_checkmate()) status = 'Checkmate!';
            else if (game.in_draw()) status = 'Draw';
            else if (game.in_stalemate()) status = 'Stalemate';

            if (game.in_check()) checkStatus = turn + ' is in Check';

            document.getElementById('turn-kpi').innerText = turn;
            document.getElementById('status-kpi').innerText = status;
            document.getElementById('check-kpi').innerText = checkStatus;

            const moves = game.history();
            document.getElementById('moves-kpi').innerText = Math.floor(moves.length / 2);

            document.getElementById('pgn-history').innerHTML = game.pgn({ max_width: 5, newline_char: '<br>' });

            let log = document.getElementById('pgn-history');
            log.scrollTop = log.scrollHeight;
        }

        function undoMove() {
            game.undo();
            board.position(game.fen());
            updateStatus();
        }

        function resetGame() {
            game.reset();
            board.start();
            updateStatus();
        }

        var config = {
            draggable: true,
            position: 'start',
            onDragStart: onDragStart,
            onDrop: onDrop,
            onSnapEnd: onSnapEnd,
            pieceTheme: 'https://chessboardjs.com/img/chesspieces/wikipedia/{piece}.png'
        };

        board = Chessboard('myBoard', config);
        updateStatus();

        $(window).resize(board.resize);
    </script>
</body>
</html>
"""

display(HTML(html_content))

In [32]:
# @title 🦖 Cyber-Dino Dash: Evolution Pro {display-mode: "form"}

import json
from IPython.display import HTML
from google.colab import output

def _report_js_error(message):
    print(f"JavaScript Error: {message}")

output.register_callback('report_js_error', _report_js_error)

html_content = """
<!DOCTYPE html>
<html>
<head>
    <link href=\"https://fonts.googleapis.com/css2?family=Orbitron:wght@400;700&family=Inter:wght@400;600&display=swap\" rel=\"stylesheet\">
    <style>
        body {
            font-family: 'Inter', sans-serif;
            background-color: #0f172a;
            margin: 0;
            padding: 20px;
            display: flex;
            justify-content: center;
        }
        .dashboard-container {
            display: grid;
            grid-template-columns: 1fr;
            gap: 20px;
            max-width: 800px;
            width: 100%;
        }
        .kpi-row {
            display: grid;
            grid-template-columns: repeat(3, 1fr);
            gap: 15px;
        }
        .card {
            background: #1e293b;
            border-radius: 16px;
            padding: 20px;
            box-shadow: 0 10px 15px -3px rgba(0,0,0,0.3);
            border: 1px solid #334155;
            color: white;
        }
        .kpi-card {
            text-align: center;
        }
        .kpi-value {
            font-family: 'Orbitron', sans-serif;
            font-size: 1.8rem;
            color: #22d3ee;
            text-shadow: 0 0 10px rgba(34, 211, 238, 0.5);
        }
        .kpi-label {
            font-size: 0.75rem;
            color: #94a3b8;
            text-transform: uppercase;
            letter-spacing: 0.1em;
        }
        .canvas-wrapper {
            position: relative;
            flex-grow: 1;
            min-height: 350px;
            background: #020617;
            border-radius: 12px;
            overflow: hidden;
            border: 2px solid #334155;
        }
        canvas {
            display: block;
            width: 100%;
            height: 100%;
        }
        .overlay {
            position: absolute;
            top: 50%;
            left: 50%;
            transform: translate(-50%, -50%);
            text-align: center;
            display: none;
        }
        .btn-restart {
            background: #22d3ee;
            color: #0f172a;
            border: none;
            padding: 12px 24px;
            border-radius: 8px;
            font-family: 'Orbitron';
            font-weight: bold;
            cursor: pointer;
            margin-top: 10px;
        }
    </style>
</head>
<body>
    <div class=\"dashboard-container\">
        <div class=\"kpi-row\">
            <div class=\"card kpi-card\">
                <div class=\"kpi-label\">Current Score</div>
                <div id=\"score-val\" class=\"kpi-value\">0</div>
            </div>
            <div class=\"card kpi-card\">
                <div class=\"kpi-label\">Dash Speed</div>
                <div id=\"speed-val\" class=\"kpi-value\">0.5x</div>
            </div>
            <div class=\"card kpi-card\">
                <div class=\"kpi-label\">High Score</div>
                <div id=\"high-val\" class=\"kpi-value\">0</div>
            </div>
        </div>

        <div class=\"card canvas-wrapper\">
            <canvas id=\"gameCanvas\"></canvas>
            <div id=\"gameOverOverlay\" class=\"overlay\">
                <h1 style=\"font-family: 'Orbitron'; color: #ef4444;\">SYSTEM CRASHED</h1>
                <button class=\"btn-restart\" onclick=\"resetGame()\">REBOOT DINO</button>
            </div>
        </div>

        <div class=\"card\" style=\"text-align:center; font-size: 0.8rem; color: #94a3b8;\">
            PRESS <b>SPACE</b> OR <b>TAP</b> TO JUMP • AVOID CACTI AND BIRDS
        </div>
    </div>

    <script>
        window.onerror = function(message) {
            google.colab.kernel.invokeFunction('report_js_error', [message], {});
        };

        const canvas = document.getElementById('gameCanvas');
        const ctx = canvas.getContext('2d');
        const scoreEl = document.getElementById('score-val');
        const speedEl = document.getElementById('speed-val');
        const highEl = document.getElementById('high-val');
        const overlay = document.getElementById('gameOverOverlay');

        let gameActive = true;
        let score = 0;
        let highScore = 0;
        let baseSpeed = 3;
        let gameSpeed = baseSpeed;
        let frameCount = 0;

        const dino = {
            x: 50,
            y: 0,
            width: 44,
            height: 47,
            dy: 0,
            jumpForce: 13,
            gravity: 0.7,
            grounded: false
        };

        let obstacles = [];

        function resize() {
            canvas.width = canvas.offsetWidth;
            canvas.height = canvas.offsetHeight;
            dino.y = canvas.height - dino.height - 20;
        }

        window.addEventListener('resize', resize);
        resize();

        function drawDino(x, y, isJumping, frame) {
            ctx.fillStyle = '#22d3ee';
            ctx.fillRect(x + 10, y, 24, 25);
            ctx.fillRect(x + 30, y, 10, 10);
            ctx.fillRect(x + 30, y + 10, 14, 5);
            ctx.fillRect(x, y + 15, 10, 15);
            ctx.fillRect(x + 10, y + 25, 20, 10);
            ctx.fillStyle = '#020617';
            ctx.fillRect(x + 32, y + 3, 3, 3);
            ctx.fillStyle = '#22d3ee';
            if (isJumping) {
                ctx.fillRect(x + 12, y + 35, 4, 8);
                ctx.fillRect(x + 24, y + 35, 4, 8);
            } else {
                const legY = (frame % 20 < 10) ? 12 : 6;
                ctx.fillRect(x + 12, y + 35, 4, legY);
                ctx.fillRect(x + 24, y + 35, 4, 18 - legY);
            }
        }

        function drawCactus(x, y, w, h) {
            ctx.fillStyle = '#f43f5e';
            ctx.fillRect(x + (w/2) - 3, y, 6, h);
            ctx.fillRect(x, y + h/3, 4, h/3);
            ctx.fillRect(x, y + (h/3)*2 - 2, w/2, 4);
            ctx.fillRect(x + w - 4, y + h/4, 4, h/3);
            ctx.fillRect(x + w/2, y + (h/4)*2 - 2, w/2, 4);
        }

        function drawBird(x, y, frame) {
            ctx.fillStyle = '#fbbf24';
            ctx.fillRect(x, y + 10, 30, 10); // Body
            ctx.fillRect(x + 25, y + 5, 10, 8); // Head
            const wingPos = (frame % 30 < 15) ? 0 : 20;
            ctx.fillRect(x + 10, y + wingPos, 10, 5); // Wing
        }

        function spawnObstacle() {
            const isBird = gameSpeed > 6 && Math.random() > 0.7;
            if (isBird) {
                obstacles.push({
                    type: 'bird',
                    x: canvas.width,
                    y: canvas.height - 90,
                    width: 35,
                    height: 20
                });
            } else {
                const h = 40 + Math.random() * 30;
                obstacles.push({
                    type: 'cactus',
                    x: canvas.width,
                    y: canvas.height - h - 20,
                    width: 25,
                    height: h
                });
            }
        }

        function update() {
            if (!gameActive) return;
            frameCount++;
            score += 0.1;
            scoreEl.innerText = Math.floor(score);

            if (frameCount % 500 === 0) {
                gameSpeed += 0.5;
                speedEl.innerText = (gameSpeed / 6).toFixed(1) + 'x';
            }

            dino.dy += dino.gravity;
            dino.y += dino.dy;
            const groundY = canvas.height - dino.height - 20;
            if (dino.y > groundY) {
                dino.y = groundY;
                dino.dy = 0;
                dino.grounded = true;
            }

            const spawnRate = Math.max(60, 150 - Math.floor(gameSpeed * 5));
            if (frameCount % spawnRate === 0) spawnObstacle();

            obstacles.forEach((obs, index) => {
                obs.x -= gameSpeed;
                if (dino.x < obs.x + obs.width - 5 && dino.x + dino.width - 5 > obs.x &&
                    dino.y < obs.y + obs.height && dino.y + dino.height > obs.y) {
                    gameOver();
                }
                if (obs.x + obs.width < 0) obstacles.splice(index, 1);
            });
        }

        function draw() {
            ctx.clearRect(0, 0, canvas.width, canvas.height);
            ctx.strokeStyle = '#334155';
            ctx.beginPath();
            ctx.moveTo(0, canvas.height - 20);
            ctx.lineTo(canvas.width, canvas.height - 20);
            ctx.stroke();

            drawDino(dino.x, dino.y, !dino.grounded, frameCount);
            obstacles.forEach(obs => {
                if (obs.type === 'bird') drawBird(obs.x, obs.y, frameCount);
                else drawCactus(obs.x, obs.y, obs.width, obs.height);
            });
        }

        function loop() {
            update();
            draw();
            requestAnimationFrame(loop);
        }

        function gameOver() {
            gameActive = false;
            if (score > highScore) highScore = Math.floor(score);
            highEl.innerText = highScore;
            overlay.style.display = 'block';
        }

        function resetGame() {
            score = 0; gameSpeed = baseSpeed; frameCount = 0; obstacles = [];
            gameActive = true; overlay.style.display = 'none';
            speedEl.innerText = '0.5x';
        }

        window.addEventListener('keydown', (e) => {
            if (e.code === 'Space' && dino.grounded && gameActive) dino.dy = -dino.jumpForce;
        });

        canvas.addEventListener('touchstart', () => {
            if (dino.grounded && gameActive) dino.dy = -dino.jumpForce;
        });

        loop();
    </script>
</body>
</html>
"""

display(HTML(html_content))
